# <u> NOTEBOOK PERMETTANT DE REPONDRE AUX DEMANDES D'ANNABELLE </U>

### <u> Import des librairies utiles pour ce projet </u>

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import statsmodels.api as sm

### <u> Chargement du dataset selon les csv remis lors de la mission </u>

In [50]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

In [51]:
print('Les données minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les données maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

Les données minimales par colonnes du dataframe sont les suivantes :
id_prod              0_0
date          2021-03-01
session_id           s_1
client_id            c_1
price               0.62
categ                  0
sex                    f
birth               1929
dtype: object

 ----------------------------------------------------------
Les données maximales par colonnes du dataframe sont les suivantes :
id_prod             2_99
date          2023-02-28
session_id       s_99999
client_id          c_999
price              300.0
categ                  2
sex                    m
birth               2004
dtype: object

 ----------------------------------------------------------
Le type de chaque colonne du dataframe est :
id_prod        object
date           object
session_id     object
client_id      object
price         float64
categ           int64
sex            object
birth           int64
dtype: object


In [52]:
# Conversion de la colonne date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

## <u> 1. Le chiffre d'affaires </U>

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [53]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [54]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['Moyenne glissante (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [55]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['Moyenne glissante (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [56]:
# Vérification de la création des colonnes de moyennes glissantes
df_ca.head(35)

,date,price,Moyenne glissante (semaine),Moyenne glissante (mois)
0,2021-03-01,16565.22,NaN,NaN
1,2021-03-02,15486.45,NaN,NaN
2,2021-03-03,15198.69,NaN,NaN
3,2021-03-04,15196.07,NaN,NaN
4,2021-03-05,17471.37,NaN,NaN
5,2021-03-06,15785.28,NaN,NaN
6,2021-03-07,14760.20,15780.47,NaN
7,2021-03-08,15679.53,15653.94,NaN
8,2021-03-09,15710.51,15685.95,NaN
9,2021-03-10,15496.87,15728.55,NaN


In [57]:
# Grahique concernant la saisonnaluté du CA. Pas de réelle saisonalité
# Interprétation limitée et tendance à une certaine stabilité - saisonnalité marquée mais pas de pics

figure = px.line(
    df_ca, 
    x='date', 
    y=['price', 'Moyenne glissante (semaine)','Moyenne glissante (mois)'], 
    color_discrete_sequence=px.colors.qualitative.Pastel,
    markers=False,
    title='Évolution du chiffre d\'affaires',
    labels={
        'date': 'Date (mois)',
        'value': 'Chiffre d’affaires (€)',
        'variable': 'Moyennes glissantes'
    }
)

figure.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure.update_traces(
    hovertemplate='%{y:.0f}€'
)

figure.show()

#### Chiffre d’affaires par catégorie + évolution dans le temps

In [58]:
# Création d'un df contenant les données agregées par mois
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='MS'), 'categ'])['price'].sum().reset_index()
df_lapage_mois['date AAAA-MM'] = df_lapage_mois['date'].dt.strftime('%Y-%m')

In [59]:
# Vérification des catégories uniques présentes dans la colonne categ
categories =df_lapage_mois.categ.unique()
print('Les catégories uniques du df sont les suivantes :', categories)

Les catégories uniques du df sont les suivantes : [0 1 2]


In [60]:
# Affichage du chiffre d'affaire par catégorie sur l'intégralité de la période du dataset
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

Le chiffre d'affaire total de la catégorie 0 est de 4 419 731 euros
Le chiffre d'affaire total de la catégorie 1 est de 4 827 657 euros
Le chiffre d'affaire total de la catégorie 2 est de 2 780 275 euros


In [61]:
figure2 = px.line(
    df_lapage_mois, 
    x='date', 
    y='price', 
    color='categ', 
    markers=True,
    title='Évolution mensuelle du chiffre d’affaires par catégorie',
    labels={
        'date': 'Date (mois)',
        'price': 'Chiffre d’affaires (€)',
        'categ': 'Catégories'
    }
)

figure2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure2.update_traces(
    hovertemplate='%{y:.0f}€'
    )


figure2.show()

## <u> 2. Les clients </U>

#### Nombre de clients par mois + évolution dans le temps

In [62]:
#Affichage du nombre de clients au total sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['client_id'].nunique()} clients distincts.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 8600 clients distincts.


In [63]:
#Affichage du nombre de clients uniques par mois
df_clients_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index(name='Nombre de clients'))
df_clients_par_mois.head()

,date,Nombre de clients
0,2021-03-01,5676
1,2021-04-01,5674
2,2021-05-01,5644
3,2021-06-01,5659
4,2021-07-01,5672


In [64]:
figure3=px.line(
    df_clients_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients (uniques)',
    labels={
        'date': 'Date (mois)'
    }
)

figure3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure3.show()

In [65]:
#Affichage du nombre de clients par mois
df_clients_total_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index(name='Nombre de clients'))
df_clients_total_par_mois.head()

,date,Nombre de clients
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [66]:
figure4=px.line(
    df_clients_total_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients',
    labels={
        'date': 'Date (mois)'
    }
)

figure4.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure4.show()

## <u> 3. Les transactions </U>

#### Nombre de transactions + évolution dans le temps

In [67]:
#Affichage du nombre de transactions totales sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['session_id'].count()} transactions.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 687534 transactions.


In [68]:
#Affichage du nombre de transactions par semaine
df_transactions_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['session_id'].count().reset_index(name='Nombre de transactions par semaine'))
df_transactions_semaine.head()

,date,Nombre de transactions par semaine
0,2021-03-07,6524
1,2021-03-14,6422
2,2021-03-21,6590
3,2021-03-28,6368
4,2021-04-04,6406


In [69]:
#Affichage du nombre de transactions par mois
df_transactions_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['session_id'].count().reset_index(name='Nombre de transactions par mois'))
df_transactions_mois.head()

,date,Nombre de transactions par mois
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [70]:
#Affichage du nombre de transactions par trimestre
df_transactions_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['session_id'].count().reset_index(name='Nombre de transactions par trimestre'))
df_transactions_trimestre.head()

### Prendre en compte le fait que la période commence le 01/03/21 donc le premier trimestre n'est pas significatif
### Le dernier trimestre de la période du dataset ne sera pas significatif non plus car il ne couvre que janvier et février 2023

,date,Nombre de transactions par trimestre
0,2021-01-01,28601
1,2021-04-01,83578
2,2021-07-01,83702
3,2021-10-01,90790
4,2022-01-01,88633


In [71]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_transactions= df_transactions_semaine.merge(df_transactions_mois, on = 'date', how='outer').sort_values('date')
df_transactions= df_transactions.merge(df_transactions_trimestre, on = 'date', how='outer').sort_values('date')
df_transactions['Nombre de transactions par mois'] = (df_transactions['Nombre de transactions par mois'].interpolate(method='linear'))
df_transactions['Nombre de transactions par trimestre'] = (df_transactions['Nombre de transactions par trimestre'].interpolate(method='linear'))
df_transactions.head()

,date,Nombre de transactions par semaine,Nombre de transactions par mois,Nombre de transactions par trimestre
0,2021-01-01,NaN,NaN,28601.000000
1,2021-03-01,NaN,28601.0,37763.833333
2,2021-03-07,6524.0,28569.4,46926.666667
3,2021-03-14,6422.0,28537.8,56089.500000
4,2021-03-21,6590.0,28506.2,65252.333333


In [72]:
figure5=px.line(
    df_transactions,
    x='date',
    y=[ 'Nombre de transactions par semaine','Nombre de transactions par mois','Nombre de transactions par trimestre'],
    markers=False,
    title='Évolution du nombre de transactions par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure5.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure5.show()

## <u> 4. Les produits </U>

#### Nombre de produits vendus + évolution dans le temps

In [73]:
# tous les produits sur toute la période, puis par mois, puis par semaine
#Affichage du nombre de produits vendus par semaine
df_lapage_prod_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par semaine'))
df_lapage_prod_semaine.head()

,date,Nombre de produits vendus par semaine
0,2021-03-07,1608
1,2021-03-14,1614
2,2021-03-21,1638
3,2021-03-28,1624
4,2021-04-04,1623


In [74]:
#Affichage du nombre de produits vendus par mois
df_lapage_prod_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par mois'))

In [75]:
#Affichage du nombre de produits vendus par trimestre
df_lapage_prod_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par trimestre'))

In [76]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_lapage_prod= df_lapage_prod_semaine.merge(df_lapage_prod_mois, on = 'date', how='outer').sort_values('date')
df_lapage_prod= df_lapage_prod.merge(df_lapage_prod_trimestre, on = 'date', how='outer').sort_values('date')
df_lapage_prod['Nombre de produits vendus par mois'] = (df_lapage_prod['Nombre de produits vendus par mois'].interpolate(method='linear'))
df_lapage_prod['Nombre de produits vendus par trimestre'] = (df_lapage_prod['Nombre de produits vendus par trimestre'].interpolate(method='linear'))
df_lapage_prod.head()

,date,Nombre de produits vendus par semaine,Nombre de produits vendus par mois,Nombre de produits vendus par trimestre
0,2021-01-01,NaN,NaN,2482.000000
1,2021-03-01,NaN,2482.0,2563.666667
2,2021-03-07,1608.0,2484.0,2645.333333
3,2021-03-14,1614.0,2486.0,2727.000000
4,2021-03-21,1638.0,2488.0,2808.666667


In [77]:
figure6=px.line(
    df_lapage_prod,
    x='date',
    y=[ 'Nombre de produits vendus par semaine','Nombre de produits vendus par mois','Nombre de produits vendus par trimestre'],
    markers=False,
    title='Évolution du nombre de produits uniques vendus par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure6.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure6.update_traces(
    hovertemplate='%{y:.0f}€'
    )

figure6.show()

### <u> Top, Flop, la répartition par catégorie</u>

### <u> ***Le Top 10*** </u> 

In [78]:
# Création d'une colonne dans le df principal, indiquant la quantité de produits vendus, par article
df_lapage['quantity'] = df_lapage.groupby('id_prod')['id_prod'].transform('count')
df_lapage.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967,341
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960,880
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988,1004
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989,1065
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956,1106


In [79]:
# Création d'un dataframe restreint contenant la colonne quantité
df_quantity = df_lapage[['id_prod', 'price', 'categ', 'quantity']]
df_quantity.head()

,id_prod,price,categ,quantity
0,0_1259,11.99,0,341
1,0_1390,19.37,0,880
2,0_1352,4.50,0,1004
3,0_1458,6.55,0,1065
4,0_1358,16.49,0,1106


In [80]:
# Eviter le warning concernant les bugs de pandas sur les copies de dataframes
df_quantity = df_quantity.copy()

df_quantity['ca_par_produit'] = df_quantity['quantity'] * df_quantity['price']

In [81]:
# Les tops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=False).head(10)
    print('\n--------------------------------------------------------------------------------------------- ')
    print(f'Le top 10 des produits les plus vendus de la catégorie {categ} est le suivant :')
    print('---------------------------------------------------------------------------------------------\n')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])



--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus vendus de la catégorie 0 est le suivant :
---------------------------------------------------------------------------------------------

       id_prod  categ  quantity  price  ca_par_produit
667486  0_1441      0      1235  18.99        23452.65
666021  0_1441      0      1235  18.99        23452.65
669747  0_1441      0      1235  18.99        23452.65
163240  0_1441      0      1235  18.99        23452.65
168178  0_1441      0      1235  18.99        23452.65
167771  0_1441      0      1235  18.99        23452.65
167423  0_1441      0      1235  18.99        23452.65
167115  0_1441      0      1235  18.99        23452.65
687249  0_1441      0      1235  18.99        23452.65
166639  0_1441      0      1235  18.99        23452.65

--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus ve

In [82]:
# Graphique du top 10 

### <u> ***Le Flop 10*** </u> 

In [83]:
# Les flops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=True).head(10)
    print(f'Le top 10 des produits les moins vendus de la catégorie {categ} est le suivant :')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])


Le top 10 des produits les moins vendus de la catégorie 0 est le suivant :
       id_prod  categ  quantity  price  ca_par_produit
85656   0_1539      0         1   0.99            0.99
41359   0_1284      0         1   1.38            1.38
293680  0_1653      0         2   0.99            1.98
497028  0_1653      0         2   0.99            1.98
131286   0_807      0         1   1.99            1.99
7437     0_541      0         1   1.99            1.99
6346    0_1601      0         1   1.99            1.99
46079   0_1728      0         1   2.27            2.27
335381  0_1498      0         1   2.48            2.48
266083   0_898      0         2   1.27            2.54
Le top 10 des produits les moins vendus de la catégorie 1 est le suivant :
       id_prod  categ  quantity  price  ca_par_produit
291939   1_420      1         2   7.12           14.24
280711   1_420      1         2   7.12           14.24
262495   1_224      1         4   4.95           19.80
612294   1_224      1    

In [84]:
#### Utiliser le subplot de plotly

## <u> 5. Le profil de nos clients </U>

### <u> SEGMENTATION DES CLIENTS : BtoB ET BtoC </u>

#### ***Il existe plusieurs moyens pour identifier les B to B :*** 

1. Classification selon le montant du CA total :    

In [85]:
# Nous identifions 4 clients ayant généré plusieurs centaines de milliers d'euros de CA

ca_par_client = df_lapage.groupby(['client_id'])['price'].sum().reset_index().sort_values('price', ascending=False)
ca_par_client.head(10)
# clients = ['c_1609','c_4958','c_6714','c_3454']

,client_id,price
677,c_1609,326039.89
4388,c_4958,290227.03
6337,c_6714,153918.60
2724,c_3454,114110.57
634,c_1570,5285.82
2513,c_3263,5276.87
1268,c_2140,5260.18
2108,c_2899,5214.05
7006,c_7319,5155.77
7715,c_7959,5135.75


In [86]:
# Création d'une colonne dans le df, précisant s'il s'agit d'un client B2B ou B2C selon le tri ci-dessus
clients = ['c_1609','c_4958','c_6714','c_3454']
df_lapage['type_client']=df_lapage['client_id'].apply(lambda x: 'B to B' if x in clients else 'B to C')
df_lapage.head(20)

# Création d'un df b2b 
df_b2b = df_lapage[df_lapage['client_id'].isin(clients)]

# Mise à jour du df principal excluant ces 4 clients
df_b2c = df_lapage[df_lapage['type_client'] == 'B to C']
df_b2c

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967,341,B to C
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960,880,B to C
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988,1004,B to C
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989,1065,B to C
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956,1106,B to C
...,...,...,...,...,...,...,...,...,...,...
687529,1_508,2023-02-28,s_348444,c_3573,21.92,1,f,1996,275,B to C
687530,2_37,2023-02-28,s_348445,c_50,48.99,2,f,1994,882,B to C
687531,1_695,2023-02-28,s_348446,c_488,26.99,1,f,1985,405,B to C
687532,0_1547,2023-02-28,s_348447,c_4848,8.99,0,m,1953,568,B to C


2. Classification selon le nombre d'exemplaires achetés, pour chaque produit :

In [87]:
df_lapage['quantity'] = (df_lapage.groupby(['id_prod', 'date', 'client_id'])['id_prod'].transform('count'))
df_lapage=df_lapage.sort_values('quantity',ascending=False)
df_lapage.loc[df_lapage['quantity'] > 1].head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
342807,0_1630,2022-02-26,s_171121,c_1609,12.48,0,m,1980,3,B to B
86967,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to C
126788,2_102,2021-07-18,s_64148,c_4958,59.14,2,m,1999,3,B to B
126321,2_102,2021-07-18,s_63848,c_4958,59.14,2,m,1999,3,B to B
86950,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to C


 ***Le fait de trier les B2B sur la quantité n'est pas cohérent car une même personne peut acheter plusieurs exemplaires d'un même ouvrage*** 
 ***(sortie de livres, achat pour offrir ...) : 425 clients***

### <u> Utilisation du df_b2c pour identifier le profil des clients </u> 

In [88]:
# classement de l'Age des clients par catégorie
df_b2c.sort_values(by=['categ', 'birth'], ascending=[True, True]).reset_index(drop=True)



,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
0,0_1563,2021-03-25,s_11322,c_8362,3.99,0,f,1929,698,B to C
1,0_1159,2021-04-07,s_17164,c_577,7.99,0,m,1929,485,B to C
2,0_1729,2021-04-15,s_20794,c_577,14.99,0,m,1929,114,B to C
3,0_655,2021-04-15,s_20794,c_577,9.99,0,m,1929,61,B to C
4,0_1626,2021-05-02,s_28633,c_577,14.02,0,m,1929,763,B to C
...,...,...,...,...,...,...,...,...,...,...
640729,2_39,2023-02-28,s_348194,c_7483,57.99,2,m,2004,915,B to C
640730,2_208,2023-02-28,s_348339,c_6464,54.87,2,m,2004,831,B to C
640731,2_155,2023-02-28,s_348357,c_1852,46.99,2,m,2004,615,B to C
640732,2_163,2023-02-28,s_348364,c_3009,68.99,2,m,2004,559,B to C


In [ ]:
df_b2c.info()

<class 'pandas.core.frame.DataFrame'>
Index: 640734 entries, 0 to 687533
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id_prod      640734 non-null  object        
 1   date         640734 non-null  datetime64[ns]
 2   session_id   640734 non-null  object        
 3   client_id    640734 non-null  object        
 4   price        640734 non-null  float64       
 5   categ        640734 non-null  int64         
 6   sex          640734 non-null  object        
 7   birth        640734 non-null  int64         
 8   quantity     640734 non-null  int64         
 9   type_client  640734 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(5)
memory usage: 53.8+ MB


In [ ]:
# Création de la colonne age, correspondant à la différence entre la date du jour et l'année de 'birth' avec un ajout systématique du 01-01 pour la date simulée 
df_b2c["age"] = pd.to_datetime(df_b2c["birth"].astype(str) + "-01-01", format="%Y-%m-%d")

In [ ]:
df_b2c

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client,age
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1970-01-01 00:00:00.000001967,341,B to C,56
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1970-01-01 00:00:00.000001960,880,B to C,56
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1970-01-01 00:00:00.000001988,1004,B to C,56
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1970-01-01 00:00:00.000001989,1065,B to C,56
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1970-01-01 00:00:00.000001956,1106,B to C,56
...,...,...,...,...,...,...,...,...,...,...,...
687529,1_508,2023-02-28,s_348444,c_3573,21.92,1,f,1970-01-01 00:00:00.000001996,275,B to C,56
687530,2_37,2023-02-28,s_348445,c_50,48.99,2,f,1970-01-01 00:00:00.000001994,882,B to C,56
687531,1_695,2023-02-28,s_348446,c_488,26.99,1,f,1970-01-01 00:00:00.000001985,405,B to C,56
687532,0_1547,2023-02-28,s_348447,c_4848,8.99,0,m,1970-01-01 00:00:00.000001953,568,B to C,56


In [ ]:
figure6 = px.histogram(
    df_b2c,
    x="age",
    color="categ",
    barmode="stack",
    nbins=20,
    title="Histogramme empilé des âges par catégorie"
)

figure6.show()


In [ ]:
# Top 20 des clients en terme de CA généré
df_b2c.sort_values(by='')

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
342807,0_1630,2022-02-26,s_171121,c_1609,12.48,0,m,1980,3,B to B
86967,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to C
126788,2_102,2021-07-18,s_64148,c_4958,59.14,2,m,1999,3,B to B
126321,2_102,2021-07-18,s_63848,c_4958,59.14,2,m,1999,3,B to B
86950,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to C
234377,1_466,2021-11-10,s_117310,c_6714,15.81,1,f,1968,3,B to B
343022,0_1630,2022-02-26,s_171202,c_1609,12.48,0,m,1980,3,B to B
15996,0_1599,2021-03-18,s_7921,c_2811,16.99,0,f,1987,3,B to C
15813,0_1599,2021-03-18,s_7813,c_2811,16.99,0,f,1987,3,B to C
235166,1_466,2021-11-10,s_117698,c_6714,15.81,1,f,1968,3,B to B


In [ ]:
# Flop 20 des clients en terme de CA généré
df_lapage.tail(20)

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
26,1_280,2021-03-01,s_18,c_443,15.59,1,f,1987,1,B to C
27,0_2026,2021-03-01,s_18,c_443,5.56,0,f,1987,1,B to C
28,0_1475,2021-03-01,s_19,c_6983,11.99,0,m,1989,1,B to C
687533,0_1398,2023-02-28,s_348435,c_3575,4.52,0,f,1981,1,B to C
14,1_503,2021-03-01,s_12,c_2505,26.99,1,f,1982,1,B to C
687503,1_445,2023-02-28,s_348432,c_7767,23.99,1,f,1960,1,B to C
687504,0_1461,2023-02-28,s_348400,c_159,11.99,0,m,1997,1,B to C
687505,0_1548,2023-02-28,s_348421,c_7526,9.66,0,f,1974,1,B to C
687506,1_282,2023-02-28,s_348425,c_2655,23.20,1,f,1992,1,B to C
687507,0_1488,2023-02-28,s_348426,c_4776,4.60,0,f,2000,1,B to C
